In [ ]:
import os, sys, math, ssl, io, pytz, numpy as np, pandas as pd, requests
from datetime import datetime, timedelta, date
from timezonefinder import TimezoneFinder
from meteostat import Stations, Hourly
from isd import Batch
from scp import SCPClient
import paramiko
import calendar
from pandas.errors import EmptyDataError




def convert_utc_to_local(df, local_tz):
    """
    Convert the datetime index of the DataFrame from UTC to a local timezone.

    Args:
    df : pandas.DataFrame
        DataFrame with a datetime index in UTC.
    local_tz : str
        A timezone string (e.g., 'America/Chicago').

    Returns:
    pandas.DataFrame
        DataFrame with datetime index converted to the specified local timezone.
    """
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        df = df.reset_index(level='station', drop=True)
        df.index = pd.to_datetime(df.index)

    if df.index.tz is None:
        df.index = df.index.tz_localize('UTC')
    
    df.index = df.index.tz_convert(local_tz)
    return df

def filter_dataframe_by_date(df, start_date, end_date, timezone=None):
    """
    Filter the DataFrame to include rows between the specified start and end dates,
    handling timezone differences appropriately.
    """
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    if timezone:
        start_date = start_date.tz_localize(timezone)
        end_date = end_date.tz_localize(timezone)
    else:
        df.index = df.index.tz_localize(None)

    return df.loc[(df.index >= start_date) & (df.index <= end_date)]

def get_parameters_MERRA2(lat, lon, year):
    api_endpoint = f"https://power.larc.nasa.gov/api/temporal/hourly/point?community=SB&parameters=&longitude={lon}&latitude={lat}&start={year}0101&end={year}1231&format=EPW"
    response = requests.get(api_endpoint)
    csv_data = io.StringIO(response.text)
    df = pd.read_csv(csv_data, skiprows=8, header=None)
    header = '\n'.join(response.text.splitlines()[:8])

    # Check if the dataframe has more than 8761 rows and truncate if necessary
    # Sometimes MERRA2 erroneously provides extra rows

    if calendar.isleap(int(year)):
        df = df.iloc[:8784]
    else:
        df = df.iloc[:8760]        


    return df, header

def merge_data(df, data):
    """
    Merge data into the DataFrame, interpolating small gaps and filling large gaps with custom values.

    Args:
        df (pd.DataFrame): The target DataFrame.
        data (dict): The source data dictionary.
    Returns:
        pd.DataFrame: The updated DataFrame.
    """
    # Custom fill values for each column
    fill_values = {
        6: 99.9,   # Dry bulb temperature
        7: 99.9,   # Dew point temperature
        8: 999,   # Relative humidity
        33: 999,   # Precipitation
        30: 999,   # Snow
        21: 999,  # Wind speed
        20: 999,  # Wind direction
        9: 999999    # Pressure
    }
    # Define the columns to process and their corresponding data keys
    columns_to_process = {
        6: 'temp',    # Dry bulb temperature
        7: 'dwpt',    # Dew point temperature
        8: 'rhum',    # Relative humidity
        33: 'prcp',   # Precipitation
        30: 'snow',   # Snow
        21: 'wspd',   # Wind speed
        20: 'wdir',   # Wind direction
        9: 'pres'     # Pressure
    }

    for col, key in columns_to_process.items():
        if not data[key].isna().all():
            # Convert the data to a pandas Series for interpolation
            series = pd.Series(list(data[key][1:]))
            
            # Identify gaps (missing values)
            is_missing = series.isna()
            
            # Find consecutive gaps
            gap_groups = is_missing.ne(is_missing.shift()).cumsum()
            gap_sizes = is_missing.groupby(gap_groups).transform('size')
            
            # Interpolate small gaps (<= 3 hours)
            series_interpolated = series.interpolate(method='linear', limit=3, limit_direction='both')
            
            # Fill large gaps (> 3 hours) with the custom fill value
            series_interpolated[gap_sizes > 3] = fill_values.get(col, None)  # Use None as default if no fill value is provided
            
            # Update the DataFrame column with the interpolated and filled data
            df[col] = series_interpolated.tolist()

    return df


# def merge_data(df, data):
    # Tdb
    if not data['temp'].isna().all():
        df[6] = list(data['temp'][1:])
    # Tdew
    if not data['dwpt'].isna().all():
        df[7] = list(data['dwpt'][1:])
    # RH
    if not data['rhum'].isna().all():
        df[8] = list(data['rhum'][1:])
    # Precep
    if not data['prcp'].isna().all():
        precipitation = list(data['prcp'][1:])
        # Convert precipitation to a pandas Series for interpolation
        precipitation_series = pd.Series(precipitation)
        
        # Identify gaps (missing values)
        is_missing = precipitation_series.isna()
        
        # Find consecutive gaps
        gap_groups = is_missing.ne(is_missing.shift()).cumsum()
        gap_sizes = is_missing.groupby(gap_groups).transform('size')
        
        # Interpolate small gaps (<= 3 hours)
        precipitation_series_interpolated = precipitation_series.interpolate(method='linear', limit=3, limit_direction='both')
        
        # Fill large gaps (> 3 hours) with 999
        precipitation_series_interpolated[gap_sizes > 3] = 999
        
        # Update df[33] with the interpolated and filled precipitation data
        df[33] = precipitation_series_interpolated.tolist()

    # Snow
    if not data['snow'].isna().all():
        df[30] = list(data['snow'][1:])
    # Wspeed
    if not data['wspd'].isna().all():
        df[21] = list(data['wspd'][1:])
    # Wdir
    if not data['wdir'].isna().all():
        df[20] = list(data['wdir'][1:])
    # P, go from hPa to Pa
    if not data['pres'].isna().all():
        df[9] = [x * 100 for x in list(data['pres'][1:])]

    return df

def check_missing_hours(year, df):
    """
    Checks for missing hours in the DataFrame's datetime index for a specified year.
    """
    full_index = pd.date_range(start=f"{year}-01-01", end=f"{year+1}-01-01", freq="H")
    missing_hours = full_index.difference(df.index)
    missing_hours_num = len(missing_hours)

    if missing_hours_num > 0:
        diffs = missing_hours.to_series().diff().dt.total_seconds().div(3600)
        largest_consecutive_group = (diffs != 1).cumsum().value_counts().max()
    else:
        largest_consecutive_group = 0

    return missing_hours_num, largest_consecutive_group

def get_noaa_merra2_data(lat, lon, year, file_type, save_folder):
    """
    Retrieves NOAA and MERRA2 data for a specific location and year.
    """
    retrieve_status = True
    data_noaa, tz, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists, incomplete_timeseries = get_data_noaa(lat, lon, year, save_folder)
    # data_noaa, tz, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists, incomplete_timeseries = get_data_noaa(lat, lon, year, save_folder)
    if epw_exists:
        df_merged = ''
        retrieve_status = False
        # distance = ''
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = ''
        epw_exists = True
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
    elif incomplete_timeseries:
        df_merged = ''
        retrieve_status = False
        # distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        wmo = ''
        epw_exists = False
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    try:
        data_noaa_tz_adj = filter_dataframe_by_date(convert_utc_to_local(data_noaa, tz), datetime(year, 1, 1), datetime(year+1, 1, 1))
    except AttributeError:
        # print("We don't have NOAA data for this location/year")
        df_merged = ''
        retrieve_status = False
        # distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        epw_exists = False
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    info_dict = {
    'timeshift': get_time_shift(tz),
    'elevation': elevation,
    'wmo': wmo,
    'station_name': station_name,
    'state': state,
    'country': country,
    'lat': latitude_station,
    'lon': longitude_station,
    'weather_file_type': file_type
    }

    data_noaa_tz_adj_h = data_noaa_tz_adj.resample('H').mean()
    data_noaa_tz_adj_h_interpolated = data_noaa_tz_adj_h.interpolate(method='linear', limit=3, limit_direction='both')
    hdd, cdd = calculate_hdd_cdd(data_noaa_tz_adj_h_interpolated, 'temp')
    try:
        df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
    except EmptyDataError:
        try:
            df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
        except EmptyDataError:
            try:
                df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
            except EmptyDataError:
                try:
                    df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
                except EmptyDataError:
                    df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)


    df_merged = merge_data(df_merra2, data_noaa_tz_adj_h_interpolated)

    # Check for empty cells in df_merged
    if df_merged.isnull().any().any():
        raise ValueError("The merged DataFrame (df_merged) contains empty cells. Stopping execution.")


    # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
    return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

def run_individual_location(lat, lon, year, file_type, save_folder, save_name):
    """
    Processes a single location, fetching data and handling errors.
    """
    # data_meteostat_merra2, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists = get_noaa_merra2_data(lat, lon, year, file_type, save_folder)
    data_meteostat_merra2, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists = get_noaa_merra2_data(lat, lon, year, file_type, save_folder)
    
    if epw_exists:
        retrieve_status = False
        # distance = ''
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        retrieve_info_closest_other_locations = True
    elif retrieve_status:
        retrieve_info_closest_other_locations = False
        #Save the EPW file
        if save_name != None:
            output_path = os.path.join(save_folder, f"{save_name.replace(' ', '_').replace('.', '_')}_{year}.epw")
        else:
            output_path = os.path.join(save_folder, f"{wmo}_{year}.epw")
        data_meteostat_merra2.to_csv(output_path, header=False, index=False)
        with open(output_path, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_path, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
    else:
        retrieve_info_closest_other_locations = False
        retrieve_status = False
        print('No data available for this location/year.')

    # return retrieve_status, distance, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations
    return retrieve_status, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations

def get_time_shift(timezone_name):
    """
    Calculates the time shift for a given timezone from UTC.
    """
    timezone = pytz.timezone(timezone_name)
    now = datetime.now(timezone)
    utc_offset = now.utcoffset()
    return int(utc_offset.total_seconds() // 3600)  # Return hours offset only

def calculate_hdd_cdd(df, temperature_column):
    """
    Calculate Heating Degree Days (HDD) and Cooling Degree Days (CDD) from hourly temperature data in Celsius.
    """
    df[temperature_column + '_F'] = df[temperature_column] * 9 / 5 + 32
    base_temperature = 65

    df['date'] = df.index.to_series().dt.date
    daily_mean_temp = df.groupby('date')[temperature_column + '_F'].mean().reset_index()
    daily_mean_temp.columns = ['date', 'mean_temp']

    daily_mean_temp['HDD'] = (base_temperature - daily_mean_temp['mean_temp']).clip(lower=0)
    daily_mean_temp['CDD'] = (daily_mean_temp['mean_temp'] - base_temperature).clip(lower=0)

    total_hdd = daily_mean_temp['HDD'].sum()
    total_cdd = daily_mean_temp['CDD'].sum()

    return int(total_hdd), int(total_cdd)

def check_epw_exists(save_folder, year, wmo):
    return os.path.exists(f'{save_folder}/{wmo}_{year}.epw')

def calc_combined_ground_temperatures(df):
    """
    Calculate shallow ground temperatures for multiple depths using the Kusuda and Achenbach model
    and format results as a single EPW GROUND TEMPERATURES line.
    This is the adapted version of the method used in ResStock:
    https://github.com/NREL/OpenStudio-HPXML/blob/master/HPXMLtoOpenStudio/resources/weather.rb#L315-L346

    Parameters:
        df (pd.DataFrame): DataFrame with hourly temperatures in column 6.

    Returns:
        str: Combined GROUND TEMPERATURES line in EPW file format for all depths.
    """
    depths = [0.5, 2, 4]  # Depths to consider (in meters)

    # Conversion utility
    def convert(value, from_unit, to_unit):
        if from_unit == "yr" and to_unit == "hr":
            return value * 365.25 * 24  # 1 year = 365.25 days * 24 hours
        elif from_unit == "C" and to_unit == "R":
            return (value + 273.15) * 9 / 5  # Celsius to Rankine
        elif from_unit == "R" and to_unit == "C":
            return (value - 491.67) * 5 / 9  # Rankine to Celsius
        elif from_unit == "C" and to_unit == "F":
            return value * 9 / 5 + 32  # Celsius to Fahrenheit
        elif from_unit == "F" and to_unit == "C":
            return (value - 32) * 5 / 9  # Fahrenheit to Celsius
        elif from_unit == "R" and to_unit == "F":
            return value - 459.67  # Rankine to Fahrenheit
        else:
            raise ValueError(f"Unsupported conversion from {from_unit} to {to_unit}")

    # Ensure proper datetime index
    df.index = pd.to_datetime({
        'year': df[0],
        'month': df[1],
        'day': df[2],
        'hour': df[3]
    })

    # Constants
    amon = [15.0, 46.0, 74.0, 95.0, 135.0, 166.0, 196.0, 227.0, 258.0, 288.0, 319.0, 349.0]  # Approx. mid-month days
    po = 0.6  # Phase offset
    dif = 0.025  # Thermal diffusivity (m²/hr)
    p = convert(1.0, 'yr', 'hr')  # Convert 1 year to hours

    # Use column 6 for temperatures
    df.rename(columns={6: 'Dry Bulb Temperature (°C)'}, inplace=True)

    # Calculate monthly and annual averages in Celsius
    monthly_avg_drybulbs_c = df.groupby(df.index.month)['Dry Bulb Temperature (°C)'].mean()
    annual_avg_drybulb_c = df['Dry Bulb Temperature (°C)'].mean()

    # Convert average temperatures to Rankine for decay calculations
    monthly_avg_drybulbs_r = monthly_avg_drybulbs_c.apply(lambda x: convert(x, 'C', 'R'))
    annual_avg_drybulb_r = convert(annual_avg_drybulb_c, 'C', 'R')

    # Prepare the combined GROUND TEMPERATURES line
    combined_ground_temperatures = ["GROUND TEMPERATURES", str(len(depths))]  # Start with header and number of depths

    for depth in depths:
        # Kusuda and Achenbach parameters
        beta = math.sqrt(math.pi / (p * dif)) * 10.0
        x = math.exp(-beta)
        s = math.sin(beta)
        c = math.cos(beta)
        y = (x**2 - 2.0 * x * c + 1.0) / (2.0 * beta**2.0)
        depth_factor = math.exp(-depth * math.sqrt(math.pi / (p * dif)))

        gm = math.sqrt(y) * depth_factor
        z = (1.0 - x * (c + s)) / (1.0 - x * (c - s))
        phi = math.atan(z)
        bo = (monthly_avg_drybulbs_r.max() - monthly_avg_drybulbs_r.min()) * 0.5

        # Calculate shallow ground temperatures
        shallow_ground_monthly_temps_r = []
        for i in range(12):  # Loop through 12 months
            theta = amon[i] * 24.0  # Day of the year to hours
            temp = annual_avg_drybulb_r - bo * math.cos(2.0 * math.pi / p * theta - po - phi) * gm
            shallow_ground_monthly_temps_r.append(temp)

        # Convert results to Celsius
        shallow_ground_monthly_temps_c = [convert(temp, 'R', 'C') for temp in shallow_ground_monthly_temps_r]

        # Add the depth, empty spaces, and monthly temperatures to the combined line
        combined_ground_temperatures.append(f"{depth:.1f}")
        combined_ground_temperatures.extend([""] * 3)  # Add three empty spaces
        combined_ground_temperatures.extend([f"{temp:.2f}" for temp in shallow_ground_monthly_temps_c])

    # Return the formatted line as a single string
    return ",".join(combined_ground_temperatures)

def create_header(df, year, info_dict):
    header_lines = []

    #Calculated parameters 
    first_day_year = pd.to_datetime(date.min.replace(year=year)).day_name()
    leap_status = lambda year: 'Yes' if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 'No'
    dst_start, dst_end = get_dst_start_end(year, info_dict['lat'], info_dict['lon'])
    design_conditions_file = 'resources/design_conditions.csv'
    design_conditions_line = find_closest_design_condition(float(info_dict['lat']), float(info_dict['lon']), design_conditions_file)
    ground_temp_line = calc_combined_ground_temperatures(df)
    #Hardcoded parameters
    number_of_holidays = 0
    number_of_data_periods = 1
    number_of_records_per_hour = 1

    # line_1
    header_lines.append(f"LOCATION,{info_dict['station_name']},{info_dict['state']},{info_dict['country']},{info_dict['weather_file_type']},{info_dict['wmo']},{info_dict['lat']},{info_dict['lon']},{info_dict['timeshift']},{info_dict['elevation']}")
    # line_2
    header_lines.append(design_conditions_line)
    # line_3
    header_lines.append(f"TYPICAL/EXTREME PERIODS,0")
    # line_4
    header_lines.append(ground_temp_line)
    # header_lines.append(f"GROUND TEMPERATURES,0")
    # header_lines.append(f"GROUND TEMPERATURES,3,.5,,,,-16.34,-17.80,-15.22,-11.16,-0.57,7.61,13.13,14.81,11.95,5.60,-2.89,-10.76,2,,,,-10.97,-13.57,-13.04,-10.89,-3.80,2.61,7.74,10.49,9.90,6.30,0.46,-5.74,4,,,,-6.53,-9.19,-9.78,-8.97,-4.96,-0.64,3.32,6.08,6.72,5.16,1.73,-2.4")
    # line_5
    try:
        header_lines.append(f"HOLIDAYS/DAYLIGHT SAVINGS,{leap_status(year)},{dst_start.month}/{dst_start.day},{dst_end.month}/{dst_end.day},{number_of_holidays}")
    except AttributeError:
        #We cannot retrieve DST dates, let's set them to 0
        header_lines.append(f"HOLIDAYS/DAYLIGHT SAVINGS,{leap_status(year)},0,0,{number_of_holidays}")
    # line_6
    header_lines.append(f"COMMENTS 1, ")
    # line_7
    header_lines.append(f"COMMENTS 2, ")
    # line_8
    header_lines.append(f"DATA PERIODS,{number_of_data_periods},{number_of_records_per_hour},Data,{first_day_year},{df.iloc[0, 1]}/{df.iloc[0, 2]},{df.iloc[-1, 1]}/{df.iloc[-1, 2]}")

    return header_lines

def fix_wmo(wmo):
    """
    Attempts to fix or standardize the WMO code format.
    """
    try:
        return str(int(wmo))
    except ValueError:
        icao = wmo
        icao_converted = get_wmo_from_icao_NOAA(icao)
        if isinstance(icao_converted, type(None)):
            return icao
        else:
            return icao_converted

    # return wmo

def get_data_noaa(lat, lon, year, save_folder):
    """
    Fetches NOAA data for a given location and year, handling timezones and missing data.
    """
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year - 1, 12, 31)
    end = datetime(year + 1, 1, 2)

    stations = Stations().nearby(lat, lon)

    epw_exists = False
    station_number = 0
    len_data = 0

    incomplete_timeseries = True
    while incomplete_timeseries:
        station_number += 1
        wmo = fix_wmo(str(stations.fetch(station_number).index.values[-1]))
        # First check if EPW already exists
        if check_epw_exists(save_folder, year, wmo):
            epw_exists = True
            incomplete_timeseries = False
            break
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        if (len(data.index) >100) & (station_number>1):
            data = data.loc[data.index.get_level_values('station').unique()[-1]]


        # Fetching hourly data for those stations
        # data = Hourly(stations.fetch(2), start, end, model=True).fetch()
        # Hourly(stations.fetch(2), start, end, model=True).fetch().loc[Hourly(stations.fetch(2), start, end, model=True).fetch().index.get_level_values('station').unique()[-1]]



        # print('+++++++')
        # print(lat)
        # print(lon)
        # print(start)
        # print(end)
        
        len_data = len(data.index)
        missing_hours_num, largest_consecutive_group = check_missing_hours(year, data)
        if (len_data > 8000) & (largest_consecutive_group <= 3):
            incomplete_timeseries = False
        # distance = stations.fetch(station_number)['distance'].values[-1]
        # print('distance')
        # print(distance)
        # print('len_data')
        # print(len(data.index))
        # print(largest_consecutive_group)
        # print(incomplete_timeseries)
        # Let's stop after 100mi
        # if distance > 16093400:
        #     break
        
    # try:
    #     if distance is not None:
    #         print('Distance')
    #         print(distance)
    # except UnboundLocalError:
    #     print("Distance variable is not defined yet.")


    if epw_exists | incomplete_timeseries:
        data = ''
        timezone = ''
        # distance = ''
        elevation = ''
        station_name = ''
        state = ''
        country = ''
        latitude_station = ''
        longitude_station = ''
                
    else:
        station_info = stations.fetch(station_number)
        timezone = station_info['timezone'].values[-1]
        elevation = station_info['elevation'].values[-1]
        # distance = stations.fetch()['distance'].values[-1]
        wmo = fix_wmo(str(station_info.index.values[-1]))
        station_name = station_info['name'].values[-1]
        state = station_info['region'].values[-1]
        country = station_info['country'].values[-1]
        latitude_station = station_info['latitude'].values[-1]
        longitude_station = station_info['longitude'].values[-1]

    # return data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries
    return data, timezone, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries

def update_if_missing(df, index, col_name, new_value):
    if pd.isna(df.at[index, col_name]) or not df.at[index, col_name]:
        df.at[index, col_name] = new_value

def retrieve_info_other_location(wmo, zipcodes, year):
    retrieve_status = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"EPW_file_name_{year}"].values[0]
    # distance = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"distance_location_station_miles_{year}"].values[0]
    hdd = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"hdd_base65F_{year}"].values[0]
    cdd = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"cdd_base65F_{year}"].values[0]
    # return retrieve_status, distance, hdd, cdd
    return retrieve_status, hdd, cdd

def get_wmo_from_icao_NOAA(icao_code):
    # Path to the local CSV file in the resource folder
    csv_file_path = os.path.join(os.path.join(os.getcwd(), 'resources'), 'isd-history.csv')

    # Read the CSV file
    try:
        with open(csv_file_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()
            headers = lines[0].split(',')
            icao_index = headers.index('"ICAO"')
            wmo_index = headers.index('"USAF"')

            for line in lines[1:]:
                fields = line.split(',')
                if fields[icao_index].strip('"') == icao_code.upper():
                    return fields[wmo_index].strip('"')

    except FileNotFoundError:
        print(f"CSV file not found at path: {csv_file_path}")
        return None
    except Exception as e:
        # print(f"An error occurred: {e}")
        return None

def get_dst_start_end(year, latitude, longitude):
    # Get the timezone for the given latitude and longitude
    tf = TimezoneFinder()
    timezone_str = tf.timezone_at(lat=latitude, lng=longitude)
    
    if timezone_str is None:
        raise ValueError("Could not find timezone for the given coordinates.")
    
    # Get the timezone object
    timezone = pytz.timezone(timezone_str)
    
    # Define the dates for the beginning and end of the year (naive datetime)
    start_of_year = datetime(year, 1, 1)
    end_of_year = datetime(year, 12, 31)
    
    dst_start = None
    dst_end = None

    # Start by localizing the first date
    previous_offset = timezone.localize(start_of_year).dst()

    # Loop through each day of the year
    for dt in [start_of_year + timedelta(days=i) for i in range((end_of_year - start_of_year).days + 1)]:
        localized_dt = timezone.localize(dt)  # Localize naive datetime
        current_offset = localized_dt.dst()
        
        if previous_offset == timedelta(0) and current_offset != timedelta(0):
            dst_start = localized_dt
        elif previous_offset != timedelta(0) and current_offset == timedelta(0):
            dst_end = localized_dt
            break
        
        previous_offset = current_offset
    
    return dst_start, dst_end

# Function to calculate the distance between two points given their latitudes and longitudes
def haversine_distance(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1 
    dlon = lon2 - lon1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    r = 6371  # Radius of Earth in kilometers. Use 3956 for miles. Determines return value units.

    #Distance returned in km
    return c * r

def find_closest_design_condition(lat,lon,design_conditions_file):
    """
    Finds the closest design condition from the CSV file based on the given latitude and longitude.

    Parameters:
    lat (float): The latitude of the target location.
    lon (float): The longitude of the target location.
    csv_file (str): The path to the CSV file containing design conditions.

    Returns:
    str: The 2021 design condition string for the closest location.
    """
    
    # Read the CSV file
    df = pd.read_csv(design_conditions_file)

    # Calculate distance from target coordinates to each row in the dataframe
    df['distance'] = df.apply(lambda row: haversine_distance(lat, lon, row['latitude'], row['longitude']), axis=1)

    # Find the row with the minimum distance
    closest_row = df.loc[df['distance'].idxmin()]

    # Return the design conditions for 2021
    return closest_row['2021_design_conditions']

def retrieve_distance_station_location(wmo, lat_location, lon_location):
    meteostat_stations = pd.read_csv('resources/meteostat_stats.csv', index_col='id')
    lat_station = meteostat_stations[meteostat_stations.index == wmo]['latitude'].values[0]
    lon_station = meteostat_stations[meteostat_stations.index == wmo]['longitude'].values[0]
    distance_km = haversine_distance(lat_location, lon_location, lat_station, lon_station)
    distance_mi = distance_km*0.621371
    return distance_mi

# Define constants
year = 2022
file_type = 'AMY'
save_folder = 'epws_wmo'

# Check if the 'zipcodes' variable is already defined
if 'zipcodes' not in globals():
    # Load the zip codes CSV only if 'zipcodes' is not already defined
    zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'EPW_file_name_{year}': str, f'weather_station_wmo_{year}': str})

# Initialize a counter for iterations
counter = 0

# Process each row in the DataFrame starting from the specified index
for index, row in zipcodes.iloc[0:].iterrows():
    # if float(row.get(f"distance_location_station_miles_{year}")) < 50:
    #     continue
    print(index)


    zip_code = str(row['zip0']).zfill(5)  # Ensure the zip code is a string and pad with leading zeros if needed
    print(zip_code)
    lat = row['lat']
    lon = row['lng']
    save_name = None

    # print(lat)
    # print(lon)
    # print(row['city'])

    # Retrieve data for the current location
    # retrieve_status, distance, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    retrieve_status, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    if retrieve_info_closest_other_locations:
        try:
            # retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
            retrieve_status, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
        except IndexError:
            print(wmo)
            # retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
            retrieve_status, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)



    # Validate that retrieve_status is a boolean
    if not isinstance(retrieve_status, bool):
        raise TypeError(f"retrieve_status is not a boolean. Actual value: {retrieve_status}. Program stopped.")
    
    distance_mi = retrieve_distance_station_location(wmo, lat, lon)

    # Update the DataFrame only if the cell is empty or contains a placeholder (like 'nan')
    update_if_missing(zipcodes, index, f"EPW_file_name_{year}", retrieve_status)
    update_if_missing(zipcodes, index, f"distance_location_station_miles_{year}", distance_mi)  # Convert from meters to miles
    update_if_missing(zipcodes, index, f"weather_station_wmo_{year}", wmo)
    update_if_missing(zipcodes, index, f"hdd_base65F_{year}", hdd)
    update_if_missing(zipcodes, index, f"cdd_base65F_{year}", cdd)

    # Increment the counter
    counter += 1

    # Every 10 iterations, save the DataFrame and reopen it
    if counter % 10 == 0:
        # Save the DataFrame to the CSV file
        zipcodes.to_csv('resources/zip_code_list.csv', index=False)

        # Reopen the file to ensure the latest version is loaded
        # zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})

# After the loop is done, ensure the latest state is saved
zipcodes.to_csv('resources/zip_code_list.csv', index=False)

# KVHN0
# KSUN
# 74611


0
99638
1
99923
2
99776
3
99566
4
99780
5
89017


ValueError: The merged DataFrame (df_merged) contains empty cells. Stopping execution.

Debug

In [3]:

def get_noaa_merra2_data(lat, lon, year, file_type, save_folder):
    """
    Retrieves NOAA and MERRA2 data for a specific location and year.
    """
    retrieve_status = True
    data_noaa, tz, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists, incomplete_timeseries = get_data_noaa(lat, lon, year, save_folder)
    # data_noaa, tz, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists, incomplete_timeseries = get_data_noaa(lat, lon, year, save_folder)
    if epw_exists:
        df_merged = ''
        retrieve_status = False
        # distance = ''
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = ''
        epw_exists = True
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
    elif incomplete_timeseries:
        df_merged = ''
        retrieve_status = False
        # distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        wmo = ''
        epw_exists = False
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    try:
        data_noaa_tz_adj = filter_dataframe_by_date(convert_utc_to_local(data_noaa, tz), datetime(year, 1, 1), datetime(year+1, 1, 1))
    except AttributeError:
        # print("We don't have NOAA data for this location/year")
        df_merged = ''
        retrieve_status = False
        # distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        epw_exists = False
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    info_dict = {
    'timeshift': get_time_shift(tz),
    'elevation': elevation,
    'wmo': wmo,
    'station_name': station_name,
    'state': state,
    'country': country,
    'lat': latitude_station,
    'lon': longitude_station,
    'weather_file_type': file_type
    }

    data_noaa_tz_adj_h = data_noaa_tz_adj.resample('H').mean()
    data_noaa_tz_adj_h_interpolated = data_noaa_tz_adj_h.interpolate(method='linear', limit=3, limit_direction='both')
    hdd, cdd = calculate_hdd_cdd(data_noaa_tz_adj_h_interpolated, 'temp')
    try:
        df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
    except EmptyDataError:
        try:
            df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
        except EmptyDataError:
            try:
                df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
            except EmptyDataError:
                try:
                    df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
                except EmptyDataError:
                    df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)


    df_merged = merge_data(df_merra2, data_noaa_tz_adj_h_interpolated)

    # Check for empty cells in df_merged
    # if df_merged.isnull().any().any():
    #     raise ValueError("The merged DataFrame (df_merged) contains empty cells. Stopping execution.")


    # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
    return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists


[df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists] = get_noaa_merra2_data(37.73096, -115.24786, 2022, 'AMY', '')





In [5]:
df_merged.to_csv('TEST_.csv')

In [ ]:
print(zipcodes.columns)

In [ ]:
meteostat_df = pd.read_csv('resources/meteostat_stats.csv')
zipcodes = pd.read_csv('resources/zip_code_list.csv')
meteostat_df


In [ ]:
zip_row.columns

In [1]:
import math

# Function to calculate the distance between two lat/lon points using the Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    # Radius of the Earth in miles
    R = 3958.8
    
    # Convert degrees to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    
    # Haversine formula
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    # Distance in miles
    return R * c


# Iterate over each row in zip_df
for idx, zip_row in zipcodes.iterrows():
    # print(idx)

    if bool(zip_row['Do we have data for 2022?']):
        wmo_code = zip_row['weather_station_wmo_2022']

        # print(wmo_code)
        
        # Look for a matching row in meteostat_df using WMO or ICAO code
        # station_row = meteostat_df[(meteostat_df['wmo'].astype(str) == wmo_code) | (meteostat_df['icao'].astype(str) == wmo_code)]
        try:
            station_row = meteostat_df[(meteostat_df['id'].astype(str) == str(wmo_code))]
        except ValueError:
            try:
                station_row = meteostat_df[(meteostat_df['id'].astype(str) == wmo_code[:4])]
            except TypeError:
                print('----------------------------')
                print(wmo_code)
                print(zip_row['zip0'])
                print(zip_row['Do we have data for 2022?'])
                continue
                # station_row = meteostat_df[(meteostat_df['icao'].astype(str) == wmo_code[:4])]
               
        



        if not station_row.empty:
            # Extract lat/lon from meteostat_df
            lat_stat = station_row['latitude'].values[0]
            lon_stat = station_row['longitude'].values[0]
            
            # Extract lat/lon from zip_df
            lat_zip = zip_row['lat']
            lon_zip = zip_row['lng']
            
            # Calculate the distance in miles
            distance = haversine(lat_zip, lon_zip, lat_stat, lon_stat)
            
            # Save the calculated distance in the new column
            zipcodes.at[idx, 'distance_location_station_miles_2022___'] = distance

# Show the updated zip_df with distances
zipcodes.head()


NameError: name 'zipcodes' is not defined

In [ ]:
meteostat_df[(meteostat_df['wmo'] == int('71345'))]

In [7]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)


In [19]:


Stations().nearby(48.72526, -111.36528).fetch().to_csv('resources/meteostat_stats.csv', index=True)

In [ ]:
def get_data_noaa(lat, lon, year, save_folder):
    """
    Fetches NOAA data for a given location and year, handling timezones and missing data.
    """
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year - 1, 12, 31)
    end = datetime(year + 1, 1, 2)

    stations = Stations().nearby(lat, lon)

    epw_exists = False
    station_number = 0
    len_data = 0

    incomplete_timeseries = True
    while incomplete_timeseries:
        station_number += 1
        wmo = fix_wmo(str(stations.fetch(station_number).index.values[-1]))
        # First check if EPW already exists
        if check_epw_exists(save_folder, year, wmo):
            epw_exists = True
            incomplete_timeseries = False
            break
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        len_data = len(data)
        missing_hours_num, largest_consecutive_group = check_missing_hours(year, data)
        if (len_data > 8000) & (largest_consecutive_group <= 3):
            incomplete_timeseries = False
        # distance = stations.fetch(station_number)['distance'].values[-1]
        # # Let's stop after 100mi
        # if distance > 160000:
        #     break

    if epw_exists | incomplete_timeseries:
        data = ''
        timezone = ''
        # distance = ''
        elevation = ''
        station_name = ''
        state = ''
        country = ''
        latitude_station = ''
        longitude_station = ''
                
    else:
        station_info = stations.fetch(station_number)
        timezone = station_info['timezone'].values[-1]
        elevation = station_info['elevation'].values[-1]
        # distance = stations.fetch()['distance'].values[-1]
        wmo = fix_wmo(str(station_info.index.values[-1]))
        station_name = station_info['name'].values[-1]
        state = station_info['region'].values[-1]
        country = station_info['country'].values[-1]
        latitude_station = station_info['latitude'].values[-1]
        longitude_station = station_info['longitude'].values[-1]

    # return data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries
    return data, timezone, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries



lat = 42.06259
lon = -72.62589

data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries = get_data_noaa(lat, lon, 2022, '')
data.head(5)

In [ ]:
data.interpolate(method='linear', limit=3, limit_direction='forward')

In [ ]:
zipcodes.head(20)

In [ ]:

file_type = 'AMY'
lat = 41.766595
lon = -88.318735
year = 2023
name= 'TEST_2023'
output_name = name + '.epw'

# Run your existing code with these parameters
data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
data_meteostat_merra2.to_csv(output_name, header=False, index=False)
with open(output_name, 'r') as original_file:
    data_content = original_file.read()
header_lines = create_header(data_meteostat_merra2, year, info_dict)
with open(output_name, 'w') as new_file:
    new_file.write("\n".join(header_lines) + "\n" + data_content)


In [ ]:
import sys
from PyQt5.QtWidgets import QApplication, QWidget, QLabel, QLineEdit, QPushButton, QVBoxLayout, QGridLayout, QMessageBox
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import requests
import ssl
import io

# Assuming all the previous functions are defined above or imported from another module


def run_individual_location(output_name, lat, lon, year, file_type, name):
    try:
        # Run your existing code with these parameters
        data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
        data_meteostat_merra2.to_csv(output_name, header=False, index=False)
        with open(output_name, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_name, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
        QMessageBox.information(window, "Success", f"Data saved successfully to {output_name}")
    except Exception as e:
        QMessageBox.critical(window, "Error", f"An error occurred: {str(e)}")


def on_run_clicked():
    lat = float(lat_input.text())
    lon = float(lon_input.text())
    year = int(year_input.text())
    file_type = file_type_input.text()
    name = name_input.text()
    output_name = output_name_input.text()
    
    run_individual_location(output_name, lat, lon, year, file_type, name)


# Initialize the application
app = QApplication(sys.argv)

# Create the main window
window = QWidget()
window.setWindowTitle("NOAA MERRA2 Data Processor")
window.setGeometry(100, 100, 400, 300)

# Create a grid layout
layout = QGridLayout()

# Add widgets for input fields
layout.addWidget(QLabel("Latitude:"), 0, 0)
lat_input = QLineEdit()
layout.addWidget(lat_input, 0, 1)
lat_input.setText("41.766595")

layout.addWidget(QLabel("Longitude:"), 1, 0)
lon_input = QLineEdit()
layout.addWidget(lon_input, 1, 1)
lon_input.setText("-88.318735")

layout.addWidget(QLabel("Year:"), 2, 0)
year_input = QLineEdit()
layout.addWidget(year_input, 2, 1)
year_input.setText("2023")

layout.addWidget(QLabel("File Type:"), 3, 0)
file_type_input = QLineEdit()
layout.addWidget(file_type_input, 3, 1)
file_type_input.setText("AMY")

layout.addWidget(QLabel("Name:"), 4, 0)
name_input = QLineEdit()
layout.addWidget(name_input, 4, 1)
name_input.setText("TEST_2023")

layout.addWidget(QLabel("Output File Name:"), 5, 0)
output_name_input = QLineEdit()
layout.addWidget(output_name_input, 5, 1)
output_name_input.setText("TEST_2023.epw")

# Add a run button
run_button = QPushButton("Run")
run_button.clicked.connect(on_run_clicked)
layout.addWidget(run_button, 6, 0, 1, 2)

# Set the layout for the main window
window.setLayout(layout)

# Show the window
window.show()

# Run the application's main loop
sys.exit(app.exec_())


## Figure out Zip Codes

In [ ]:
import pandas as pd
import numpy as np

# Define a function to calculate the Haversine distance between two points in km
def haversine(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula to calculate the distance
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of Earth in kilometers
    return c * r

# 1) Open the file resources/zip_code_list.csv as a dataframe and call it zipcodes
zipcodes = pd.read_csv('resources/zip_codes_list.csv')

# 2) Open design_conditions.csv as a dataframe and call it dc
dc = pd.read_csv('resources/meteostat_stats.csv')

# Ensure latitude and longitude columns are in float format
zipcodes['lat'] = zipcodes['lat'].astype(float)
zipcodes['lng'] = zipcodes['lng'].astype(float)
dc['latitude'] = dc['latitude'].astype(float)
dc['longitude'] = dc['longitude'].astype(float)

# 3) Initialize columns in zipcodes dataframe for storing results
zipcodes['Distance'] = np.nan
zipcodes['Location'] = ""

# 4) Loop through all the rows in zipcodes
for idx, row in zipcodes.iterrows():
    lat1 = row['lat']
    lon1 = row['lng']

    # Calculate the distance to each location in the dc dataframe
    dc['Distance'] = dc.apply(lambda x: haversine(lat1, lon1, x['latitude'], x['longitude']), axis=1)

    # 5) Find the closest location in dc
    closest_location = dc.loc[dc['Distance'].idxmin()]

    # 6) Update the zipcodes dataframe with the closest location's details
    zipcodes.at[idx, 'Distance'] = closest_location['Distance']
    zipcodes.at[idx, 'Location'] = closest_location['name']

# Display the updated dataframe
zipcodes.to_csv('resources/updated_zip_code_list_again.csv', index=False)

In [ ]:
# Ensure the 'zip' column is a string and pad with zeros to make it 5 digits
zipcodes['zip0'] = zipcodes['zip'].astype(str).str.zfill(5)

zipcodes


In [ ]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)

In [ ]:
import pandas as pd
import requests
from io import StringIO

# URL of the dataset containing ZIP codes and their respective coordinates
url = "https://raw.githubusercontent.com/scpike/us-state-county-zip/master/geo-data.csv"

# Fetching the CSV file from the URL
response = requests.get(url)
response.raise_for_status()  # Raises an error for bad responses

# Reading the CSV data into a pandas DataFrame
data = pd.read_csv(StringIO(response.text))


data.to_csv('resources/zip_codes_list.csv')


In [ ]:
import pandas as pd
import requests
from io import BytesIO
from zipfile import ZipFile

# URL of a dataset containing ZIP codes, latitude, and longitude
url = "https://simplemaps.com/static/data/us-zips/1.74/basic/simplemaps_uszips_basicv1.74.zip"

# Download the ZIP file with SSL verification disabled
response = requests.get(url, verify=False)  # Bypass SSL certificate verification
response.raise_for_status()  # Check if the request was successful

# Unzip the file and read the CSV
with ZipFile(BytesIO(response.content)) as zip_file:
    # Extract the CSV file within the ZIP
    with zip_file.open('uszips.csv') as file:
        zip_code_data = pd.read_csv(file)

# # Display the DataFrame to verify the contents
# print(zip_code_data.head())

# # Save the DataFrame to a local CSV file
zip_code_data.to_csv('resources/zip_codes_list.csv', index=False)
# print("Data saved to 'us_zip_codes_with_coordinates.csv'.")


In [ ]:
zip_code_data